In [2]:
# ============================================================
# 11. Model Efficiency
#
# 목적:
# 1. BERT와 DistilBERT 성능 비교
# 2. 파라미터 수 및 모델 가중치 크기 측정
# 3. 추론 지연시간 및 처리량 측정
# 4. GPU 메모리 사용량 측정
# 5. CSV, JSON, PNG 결과 저장
#
# 평가 데이터: validation.csv
# 테스트 데이터는 사용하지 않음
# ============================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from safetensors.torch import load_file
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertForSequenceClassification,
)


# ============================================================
# 1. 기본 설정
# ============================================================

SEED = 42
NUM_LABELS = 6
MAX_LENGTH = 128

# 추론 비교용 배치 크기
BATCH_SIZES = [1, 16, 32]

# GPU 초기 워밍업에 사용할 배치 수
WARMUP_BATCHES = 3

# 동일한 검증 데이터 반복 측정 횟수
REPEAT_RUNS = 3

# 모델 로딩에 사용한 BERT 사전학습 모델
BERT_BASE_MODEL = "google-bert/bert-base-uncased"

# 프로젝트 루트에서 실행한다고 가정
PROJECT_ROOT = Path.cwd()

# 검증 데이터
VALIDATION_PATH = (
    PROJECT_ROOT
    / "results"
    / "splits"
    / "validation.csv"
)

# Stage 08의 BERT 기준 모델 체크포인트
BERT_CHECKPOINT = (
    PROJECT_ROOT
    / "results"
    / "augmentation"
    / "wordnet_comparison"
    / "baseline"
    / "best_model.safetensors"
)

# BERT tokenizer 저장 경로
# 경로가 없으면 BERT 기본 tokenizer를 사용
BERT_TOKENIZER_PATH = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "bert_tokenizer"
)

# DistilBERT 모델 저장 디렉터리
DISTILBERT_MODEL_PATH = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "distilbert_model"
)

# 결과 저장 디렉터리
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "model_efficiency"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# GPU 사용 가능 여부 확인
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 70)
print("11단계: Model Efficiency")
print("=" * 70)

print("프로젝트 경로:", PROJECT_ROOT)
print("실행 장치:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("최대 토큰 길이:", MAX_LENGTH)
print("측정 배치 크기:", BATCH_SIZES)
print("반복 측정 횟수:", REPEAT_RUNS)


# ============================================================
# 2. 파일 존재 여부 확인
# ============================================================

required_paths = [
    VALIDATION_PATH,
    BERT_CHECKPOINT,
    DISTILBERT_MODEL_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    print("\n[오류] 다음 파일 또는 디렉터리를 찾지 못했습니다.")

    for path in missing_paths:
        print(" -", path)

    raise FileNotFoundError(
        "\n코드 상단의 경로 설정을 실제 프로젝트 구조에 맞게 수정하세요."
    )

if not DISTILBERT_MODEL_PATH.is_dir():
    raise ValueError(
        "DISTILBERT_MODEL_PATH는 저장된 모델 디렉터리여야 합니다."
    )

if not BERT_CHECKPOINT.is_file():
    raise ValueError(
        "BERT_CHECKPOINT는 .safetensors 파일이어야 합니다."
    )


# ============================================================
# 3. 검증 데이터 불러오기
# ============================================================

print("\n[1] 검증 데이터 로드")

df = pd.read_csv(VALIDATION_PATH)

required_columns = {"text", "label"}

if not required_columns.issubset(df.columns):
    raise ValueError(
        f"CSV에는 다음 컬럼이 필요합니다: {required_columns}"
    )

df = df.dropna(
    subset=["text", "label"]
).copy()

df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

if not df["label"].isin(range(NUM_LABELS)).all():
    raise ValueError(
        "label 값이 0~5 범위를 벗어났습니다."
    )

texts = df["text"].tolist()
y_true = df["label"].to_numpy()

print("검증 데이터 수:", len(df))

print("\n클래스별 데이터 수")
print(
    df["label"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 4. 모델 로드 함수
# ============================================================

def load_bert_model():
    """
    Stage 08의 BERT 기준 모델을 불러온다.

    사전학습 BERT 구조를 생성한 다음
    프로젝트에서 저장한 가중치를 엄격하게 로드한다.
    """

    print("\nBERT 모델 초기화")

    model = BertForSequenceClassification.from_pretrained(
        BERT_BASE_MODEL,
        num_labels=NUM_LABELS,
    )

    print("BERT 체크포인트 로드")

    state_dict = load_file(
        str(BERT_CHECKPOINT)
    )

    # 모델 구조와 체크포인트의 키가 일치해야 한다.
    model.load_state_dict(
        state_dict,
        strict=True,
    )

    tokenizer_path = BERT_TOKENIZER_PATH

    if not tokenizer_path.exists():
        tokenizer_path = Path(BERT_BASE_MODEL)

    tokenizer = AutoTokenizer.from_pretrained(
        str(tokenizer_path)
    )

    return model, tokenizer


def load_distilbert_model():
    """
    기존에 저장한 DistilBERT 모델과 tokenizer를 로드한다.
    """

    print("\nDistilBERT 모델 로드")

    model = AutoModelForSequenceClassification.from_pretrained(
        str(DISTILBERT_MODEL_PATH)
    )

    # 모델 디렉터리에 tokenizer가 함께 저장된 경우 우선 사용
    tokenizer_path = DISTILBERT_MODEL_PATH

    separate_tokenizer_path = (
        PROJECT_ROOT
        / "results"
        / "checkpoints"
        / "distilbert_tokenizer"
    )

    if separate_tokenizer_path.exists():
        tokenizer_path = separate_tokenizer_path

    tokenizer = AutoTokenizer.from_pretrained(
        str(tokenizer_path)
    )

    if model.config.num_labels != NUM_LABELS:
        raise ValueError(
            f"DistilBERT 출력 클래스 수가 "
            f"{model.config.num_labels}입니다. "
            f"기대값은 {NUM_LABELS}입니다."
        )

    return model, tokenizer


# ============================================================
# 5. 파라미터 수 및 가중치 파일 크기
# ============================================================

def count_parameters(model):
    """
    전체 파라미터와 학습 가능한 파라미터를 계산한다.
    """

    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return total_params, trainable_params


def get_weight_size(path):
    """
    모델 가중치 파일 크기의 합계를 계산한다.

    tokenizer, config 등은 제외한다.
    """

    path = Path(path)

    if path.is_file():
        return path.stat().st_size

    weight_suffixes = {
        ".safetensors",
        ".bin",
        ".pt",
        ".pth",
    }

    return sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
        and file.suffix in weight_suffixes
    )


# ============================================================
# 6. 검증 데이터 토큰화
# ============================================================

def tokenize_all(tokenizer, texts):
    """
    전체 검증 데이터를 미리 토큰화한다.

    이 시간은 모델의 순전파 시간과 별도로 기록한다.
    """

    print("\n검증 데이터 토큰화")

    start_time = time.perf_counter()

    encoded = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=True,
    )

    elapsed = time.perf_counter() - start_time

    print(f"토큰화 시간: {elapsed:.3f}초")

    return encoded, elapsed


def make_batch(
    encoded,
    start_idx,
    end_idx,
    tokenizer,
    device,
):
    """
    토큰화된 문장들을 배치 단위로 묶고
    해당 배치에서 가장 긴 문장에 맞춰 패딩한다.
    """

    batch_input_ids = encoded["input_ids"][
        start_idx:end_idx
    ]

    batch_attention_masks = encoded["attention_mask"][
        start_idx:end_idx
    ]

    batch = tokenizer.pad(
        {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_masks,
        },
        padding=True,
        return_tensors="pt",
    )

    return {
        key: value.to(device)
        for key, value in batch.items()
    }


# ============================================================
# 7. GPU 동기화 함수
# ============================================================

def synchronize():
    """
    GPU 연산은 비동기 실행될 수 있으므로
    시간 측정 전후에 연산 완료를 기다린다.
    """

    if torch.cuda.is_available():
        torch.cuda.synchronize()


# ============================================================
# 8. 모델 효율성 측정
# ============================================================

def benchmark_model(
    model_name,
    model,
    tokenizer,
    model_size_bytes,
):
    """
    한 모델에 대해 다음을 측정한다.

    - 파라미터 수
    - 가중치 파일 크기
    - Accuracy 및 F1 지표
    - 배치별 추론 지연시간
    - 초당 처리량
    - GPU 최대 메모리 할당량

    추론 시간은 모델 forward 실행 시간 기준이며,
    토큰화와 배치 생성 시간은 포함하지 않는다.
    """

    print("\n" + "=" * 70)
    print("모델:", model_name)
    print("=" * 70)

    model = model.to(DEVICE)
    model.eval()

    total_params, trainable_params = count_parameters(
        model
    )

    encoded, tokenization_seconds = tokenize_all(
        tokenizer,
        texts,
    )

    print("전체 파라미터 수:", f"{total_params:,}")
    print("학습 가능 파라미터 수:", f"{trainable_params:,}")

    print(
        "가중치 파일 크기:",
        f"{model_size_bytes / (1024 ** 2):.2f} MB",
    )

    model_results = []

    # --------------------------------------------------------
    # 배치 크기별 측정
    # --------------------------------------------------------

    for batch_size in BATCH_SIZES:

        print("\n" + "-" * 60)
        print("배치 크기:", batch_size)

        # GPU 메모리 측정 초기화
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        # ----------------------------------------------------
        # 8-1. Warm-up
        # ----------------------------------------------------

        warmup_end = min(
            len(texts),
            WARMUP_BATCHES * batch_size,
        )

        with torch.inference_mode():

            for start_idx in range(
                0,
                warmup_end,
                batch_size,
            ):
                end_idx = min(
                    start_idx + batch_size,
                    warmup_end,
                )

                batch = make_batch(
                    encoded,
                    start_idx,
                    end_idx,
                    tokenizer,
                    DEVICE,
                )

                _ = model(**batch)

        synchronize()

        # ----------------------------------------------------
        # 8-2. 반복 추론 측정
        # ----------------------------------------------------

        total_times_per_run = []
        predictions_per_run = []

        for repeat_idx in range(REPEAT_RUNS):

            run_forward_times = []
            run_predictions = []

            with torch.inference_mode():

                for start_idx in range(
                    0,
                    len(texts),
                    batch_size,
                ):
                    end_idx = min(
                        start_idx + batch_size,
                        len(texts),
                    )

                    batch = make_batch(
                        encoded,
                        start_idx,
                        end_idx,
                        tokenizer,
                        DEVICE,
                    )

                    # 배치 생성 및 CPU -> GPU 전송은
                    # 모델 forward 시간에서 제외한다.
                    synchronize()

                    start_time = time.perf_counter()

                    outputs = model(**batch)

                    synchronize()

                    elapsed = (
                        time.perf_counter()
                        - start_time
                    )

                    run_forward_times.append(
                        elapsed
                    )

                    predictions = (
                        outputs.logits
                        .argmax(dim=-1)
                        .cpu()
                        .numpy()
                    )

                    run_predictions.extend(
                        predictions.tolist()
                    )

            total_times_per_run.append(
                sum(run_forward_times)
            )

            predictions_per_run.append(
                np.asarray(run_predictions)
            )

            print(
                f"반복 {repeat_idx + 1}/{REPEAT_RUNS}: "
                f"{sum(run_forward_times):.3f}초"
            )

        # ----------------------------------------------------
        # 8-3. 지연시간 및 처리량 계산
        # ----------------------------------------------------

        mean_total_seconds = float(
            np.mean(total_times_per_run)
        )

        mean_batch_latency_seconds = (
            mean_total_seconds
            / np.ceil(len(texts) / batch_size)
        )

        throughput = (
            len(texts) / mean_total_seconds
            if mean_total_seconds > 0
            else float("nan")
        )

        estimated_seconds_per_sample = (
            mean_total_seconds / len(texts)
        )

        # 첫 번째 반복에서 나온 예측으로 성능 평가
        y_pred = predictions_per_run[0]

        # ----------------------------------------------------
        # 8-4. 분류 성능 계산
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_true,
            y_pred,
        )

        macro_precision = precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )

        macro_recall = recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )

        # ----------------------------------------------------
        # 8-5. GPU 메모리 계산
        # ----------------------------------------------------

        if torch.cuda.is_available():

            peak_memory_mb = (
                torch.cuda.max_memory_allocated()
                / (1024 ** 2)
            )

        else:
            peak_memory_mb = np.nan

        # ----------------------------------------------------
        # 8-6. 결과 저장용 딕셔너리
        # ----------------------------------------------------

        result = {
            "model": model_name,
            "device": str(DEVICE),
            "batch_size": batch_size,
            "max_length": MAX_LENGTH,
            "validation_samples": len(texts),

            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "model_weight_mb": (
                model_size_bytes / (1024 ** 2)
            ),

            "tokenization_seconds": tokenization_seconds,

            "mean_total_inference_seconds": (
                mean_total_seconds
            ),

            "mean_batch_latency_ms": (
                mean_batch_latency_seconds * 1000
            ),

            "estimated_seconds_per_sample": (
                estimated_seconds_per_sample
            ),

            "throughput_samples_per_second": (
                throughput
            ),

            "gpu_peak_memory_mb": peak_memory_mb,

            "accuracy": accuracy,
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
        }

        model_results.append(result)

        # ----------------------------------------------------
        # 8-7. 터미널 출력
        # ----------------------------------------------------

        print(
            f"평균 전체 추론 시간: "
            f"{mean_total_seconds:.3f}초"
        )

        print(
            f"평균 배치 지연시간: "
            f"{mean_batch_latency_seconds * 1000:.3f} ms"
        )

        print(
            f"처리량: {throughput:.2f} samples/sec"
        )

        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro Precision: {macro_precision:.4f}")
        print(f"Macro Recall: {macro_recall:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(f"Weighted F1: {weighted_f1:.4f}")

        if torch.cuda.is_available():
            print(
                f"GPU 최대 메모리 할당량: "
                f"{peak_memory_mb:.2f} MB"
            )

    # --------------------------------------------------------
    # 메모리 정리
    # --------------------------------------------------------

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model_results


# ============================================================
# 9. BERT 측정
# ============================================================

print("\n\n[1/2] BERT 측정 시작")

bert_model, bert_tokenizer = load_bert_model()

bert_size_bytes = get_weight_size(
    BERT_CHECKPOINT
)

bert_results = benchmark_model(
    model_name="BERT-base",
    model=bert_model,
    tokenizer=bert_tokenizer,
    model_size_bytes=bert_size_bytes,
)

del bert_model
del bert_tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 10. DistilBERT 측정
# ============================================================

print("\n\n[2/2] DistilBERT 측정 시작")

distilbert_model, distilbert_tokenizer = (
    load_distilbert_model()
)

distilbert_size_bytes = get_weight_size(
    DISTILBERT_MODEL_PATH
)

distilbert_results = benchmark_model(
    model_name="DistilBERT",
    model=distilbert_model,
    tokenizer=distilbert_tokenizer,
    model_size_bytes=distilbert_size_bytes,
)

del distilbert_model
del distilbert_tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 11. 결과 통합 및 저장
# ============================================================

print("\n\n" + "=" * 70)
print("측정 결과 저장")
print("=" * 70)

all_results = (
    bert_results
    + distilbert_results
)

results_df = pd.DataFrame(
    all_results
)

csv_path = (
    OUTPUT_DIR
    / "efficiency_comparison.csv"
)

json_path = (
    OUTPUT_DIR
    / "efficiency_comparison.json"
)

results_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
)

results_df.to_json(
    json_path,
    orient="records",
    force_ascii=False,
    indent=2,
)


# ============================================================
# 12. 결과 요약 출력
# ============================================================

print("\n[효율성 측정 결과]")

summary_columns = [
    "model",
    "batch_size",
    "total_parameters",
    "model_weight_mb",
    "mean_total_inference_seconds",
    "mean_batch_latency_ms",
    "throughput_samples_per_second",
    "gpu_peak_memory_mb",
    "accuracy",
    "macro_f1",
    "weighted_f1",
]

print(
    results_df[summary_columns].to_string(
        index=False
    )
)

print("\nCSV 저장:", csv_path)
print("JSON 저장:", json_path)


# ============================================================
# 13. 그래프 생성 함수
# ============================================================

def save_bar_chart(
    data,
    x_column,
    y_column,
    title,
    xlabel,
    ylabel,
    output_path,
    rotation=15,
):
    """
    막대그래프 생성 및 저장
    """

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.bar(
        data[x_column].astype(str),
        data[y_column],
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    ax.tick_params(
        axis="x",
        rotation=rotation,
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(fig)


# ============================================================
# 14. 모델별 대표 결과 추출
# ============================================================

# 단일 요청 지연시간 비교
batch1_df = results_df[
    results_df["batch_size"] == 1
].copy()

# 배치 처리량 비교
batch32_df = results_df[
    results_df["batch_size"] == 32
].copy()

# 모델마다 중복되지 않는 크기 정보
model_size_df = (
    results_df[
        ["model", "model_weight_mb"]
    ]
    .drop_duplicates(
        subset=["model"]
    )
)


# ============================================================
# 15. Macro F1 비교 그래프
# ============================================================

save_bar_chart(
    data=batch1_df,
    x_column="model",
    y_column="macro_f1",
    title="Validation Macro F1 Comparison",
    xlabel="Model",
    ylabel="Macro F1",
    output_path=(
        OUTPUT_DIR
        / "macro_f1_comparison.png"
    ),
)


# ============================================================
# 16. 단일 요청 지연시간 그래프
# ============================================================

save_bar_chart(
    data=batch1_df,
    x_column="model",
    y_column="mean_batch_latency_ms",
    title="Single-Request Inference Latency",
    xlabel="Model",
    ylabel="Latency (ms)",
    output_path=(
        OUTPUT_DIR
        / "single_request_latency.png"
    ),
)


# ============================================================
# 17. 처리량 비교 그래프
# ============================================================

save_bar_chart(
    data=batch32_df,
    x_column="model",
    y_column="throughput_samples_per_second",
    title="Batch Size 32 Throughput",
    xlabel="Model",
    ylabel="Samples per Second",
    output_path=(
        OUTPUT_DIR
        / "throughput_comparison.png"
    ),
)


# ============================================================
# 18. 모델 가중치 크기 비교 그래프
# ============================================================

save_bar_chart(
    data=model_size_df,
    x_column="model",
    y_column="model_weight_mb",
    title="Model Weight Size Comparison",
    xlabel="Model",
    ylabel="Weight Size (MB)",
    output_path=(
        OUTPUT_DIR
        / "model_size_comparison.png"
    ),
)


# ============================================================
# 19. 성능과 처리량 관계 그래프
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.scatter(
    batch32_df[
        "throughput_samples_per_second"
    ],
    batch32_df["macro_f1"],
)

for _, row in batch32_df.iterrows():

    ax.annotate(
        row["model"],
        (
            row[
                "throughput_samples_per_second"
            ],
            row["macro_f1"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

ax.set_title(
    "Validation Macro F1 vs Throughput"
)

ax.set_xlabel(
    "Throughput (samples/sec)"
)

ax.set_ylabel(
    "Validation Macro F1"
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "f1_vs_throughput.png",
    dpi=200,
    bbox_inches="tight",
)

plt.close(fig)


# ============================================================
# 20. 그래프 생성 완료
# ============================================================

print("\n" + "=" * 70)
print("11단계 Model Efficiency 완료")
print("=" * 70)

print("\n결과 저장 경로:")
print(OUTPUT_DIR)

print("\n생성된 파일:")

for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(" -", path.name)

11단계: Model Efficiency
프로젝트 경로: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis
실행 장치: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
최대 토큰 길이: 128
측정 배치 크기: [1, 16, 32]
반복 측정 횟수: 3

[1] 검증 데이터 로드
검증 데이터 수: 65422

클래스별 데이터 수
label
0    18986
1    22211
2     5377
3     9013
4     7505
5     2330
Name: count, dtype: int64


[1/2] BERT 측정 시작

BERT 모델 초기화


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT 체크포인트 로드

모델: BERT-base

검증 데이터 토큰화
토큰화 시간: 1.392초
전체 파라미터 수: 109,486,854
학습 가능 파라미터 수: 109,486,854
가중치 파일 크기: 417.68 MB

------------------------------------------------------------
배치 크기: 1
반복 1/3: 165.072초
반복 2/3: 166.465초
반복 3/3: 166.973초
평균 전체 추론 시간: 166.170초
평균 배치 지연시간: 2.540 ms
처리량: 393.71 samples/sec
Accuracy: 0.9430
Macro Precision: 0.8921
Macro Recall: 0.9566
Macro F1: 0.9182
Weighted F1: 0.9448
GPU 최대 메모리 할당량: 852.77 MB

------------------------------------------------------------
배치 크기: 16
반복 1/3: 38.829초
반복 2/3: 39.158초
반복 3/3: 39.046초
평균 전체 추론 시간: 39.011초
평균 배치 지연시간: 9.540 ms
처리량: 1677.03 samples/sec
Accuracy: 0.9430
Macro Precision: 0.8921
Macro Recall: 0.9566
Macro F1: 0.9182
Weighted F1: 0.9448
GPU 최대 메모리 할당량: 913.92 MB

------------------------------------------------------------
배치 크기: 32
반복 1/3: 35.252초
반복 2/3: 35.259초
반복 3/3: 35.314초
평균 전체 추론 시간: 35.275초
평균 배치 지연시간: 17.249 ms
처리량: 1854.64 samples/sec
Accuracy: 0.9430
Macro Precision: 0.8921
Macro Recall: 0.956

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


모델: DistilBERT

검증 데이터 토큰화
토큰화 시간: 1.064초
전체 파라미터 수: 66,958,086
학습 가능 파라미터 수: 66,958,086
가중치 파일 크기: 255.44 MB

------------------------------------------------------------
배치 크기: 1
반복 1/3: 97.897초
반복 2/3: 90.714초
반복 3/3: 90.591초
평균 전체 추론 시간: 93.067초
평균 배치 지연시간: 1.423 ms
처리량: 702.95 samples/sec
Accuracy: 0.9426
Macro Precision: 0.9338
Macro Recall: 0.8896
Macro F1: 0.9053
Weighted F1: 0.9412
GPU 최대 메모리 할당량: 690.53 MB

------------------------------------------------------------
배치 크기: 16
반복 1/3: 19.772초
반복 2/3: 19.905초
반복 3/3: 19.926초
평균 전체 추론 시간: 19.868초
평균 배치 지연시간: 4.859 ms
처리량: 3292.91 samples/sec
Accuracy: 0.9426
Macro Precision: 0.9338
Macro Recall: 0.8896
Macro F1: 0.9053
Weighted F1: 0.9412
GPU 최대 메모리 할당량: 751.68 MB

------------------------------------------------------------
배치 크기: 32
반복 1/3: 17.522초
반복 2/3: 17.588초
반복 3/3: 17.594초
평균 전체 추론 시간: 17.568초
평균 배치 지연시간: 8.591 ms
처리량: 3723.92 samples/sec
Accuracy: 0.9426
Macro Precision: 0.9338
Macro Recall: 0.8896
Macro F1: 0.9053
W